# Superstore - Retail

#### Data pre-processing

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")
data.shape

(9994, 21)

In [3]:
data.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [4]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [5]:
data.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [81]:
# Creating unique data tables
data["product_id"] = [x.split("-")[2] for x in data["Product ID"].values]
customer = data[["Customer ID", "Customer Name", "Segment", "Country","Region", "State", "City", "Postal Code", ]].drop_duplicates().reset_index(drop=True)
product = data[["product_id","Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year
dates['quarter'] = dates['date'].dt.quarter


product_price_history = data[["product_id", "Product ID","Category","Sub-Category", "Discount", "Sales","Quantity",
        "Profit"]].drop_duplicates().merge(orders, 
                        on="Product ID").merge(
                            dates, 
                            left_on="Order Date", 
                            right_on="date")[["product_id", "Category","Sub-Category",
                                                "year","month_num", "month",
                                                "Discount_x",
                                                "Sales_x","Quantity_x","Profit_x"]].drop_duplicates().sort_values(
                     ["year","month_num", "product_id"]).reset_index(drop=True)


product_price = product_price_history[product_price_history["Discount_x"] == 0.0]
product_price["Sales_x"] = round(product_price["Sales_x"] / product_price_history["Quantity_x"],2)
product_price["Profit_x"] = round(product_price["Profit_x"] / product_price_history["Quantity_x"],2)
product_price["cost"] = round(product_price["Sales_x"] - product_price["Profit_x"],2)
product_price= product_price[["product_id", "Category","Sub-Category","year","month","cost"]].drop_duplicates().reset_index(drop=True)
product_price.columns = ["product_id","category","sub_category","year","month","cost"]

product_price_disc = product_price_history[product_price_history["Discount_x"] != 0.0]
product_price_disc["cost"] = round(product_price_disc["Sales_x"] - product_price_disc["Profit_x"],2)
product_price_disc= product_price_disc[["product_id","year","month", "Sales_x","Discount_x", "Profit_x","cost"]].drop_duplicates().reset_index(drop=True)
product_price_disc.columns = ["product_id","year","month","revenue", "discount", "profit","cost"]



In [74]:
# creating aggregated view of orders
columns = ['Customer ID', 'Order ID', 'Order Date', 'year',"quarter",  'month', "month_num",'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Customer ID", "Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","quarter", "month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg = orders_agg.sort_values('Order Date').reset_index(drop=True)
orders_agg.columns = ["customer_id", 'order_id', 'order_date', 'year',"quarter",'month', "month_num",'ship_mode', 'days_till_shipped', 'revenue',
       'quantity', 'products' ]
orders_agg

,customer_id,order_id,order_date,year,quarter,month,month_num,ship_mode,days_till_shipped,revenue,quantity,products
0,BD-11500,CA-2014-140795,2014-01-02,2014,1,January,1,First Class,59,468.900,6,1
1,GW-14605,CA-2014-168312,2014-01-03,2014,1,January,1,Standard Class,181,513.861,6,2
2,VF-21715,CA-2014-113880,2014-01-03,2014,1,January,1,Standard Class,120,651.588,9,2
3,HR-14770,US-2014-143707,2014-01-03,2014,1,January,1,Standard Class,120,5.940,3,1
4,DB-13060,CA-2014-104269,2014-01-03,2014,1,January,1,Second Class,151,457.568,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5004,MC-17845,US-2017-102638,2017-12-29,2017,4,December,12,First Class,2,6.030,3,1
5005,CC-12430,CA-2017-126221,2017-12-30,2017,4,December,12,Standard Class,122,209.300,2,1
5006,JM-15580,CA-2017-156720,2017-12-30,2017,4,December,12,Standard Class,61,3.024,3,1
5007,PO-18865,CA-2017-143259,2017-12-30,2017,4,December,12,Standard Class,61,466.842,14,3


In [8]:
# Let's check total  orders
print("Total orders:", len(orders_agg['order_id']))

# Let's check total  sales
print("Total sales:", round(orders_agg['revenue'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


#### Business Goal - Increase revenue

Are sales growing every year?

In [9]:
# Total orders, sales, products, quantities through 2014-2017
orders_rev = (orders_agg.groupby(["year"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year"])["products"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["quantity"].sum(), 
    how="left", 
    on="year").join(
    orders_agg.groupby(["year"])["revenue"].sum(), 
    how="left", 
    on="year")
orders_rev["%_change"] = round(orders_rev["revenue"].pct_change(),2)
orders_rev

,year,order_id,products,quantity,revenue,%_change
0,2014,969,1992,7579,483966.1261,NaN
1,2015,1038,2102,7979,470532.5090,-0.03
2,2016,1315,2587,9837,609205.5980,0.29
3,2017,1687,3312,12476,733215.2552,0.20


There has been a steady increase in number of orders placed, products sold and quantities. However, if you look at revenue, it decreased in 2015 by 3% but experienced a ~30% jump in 2016 and 20% jump in 2017

Yearly-revenue wise things look steady.

Looking at the revenue quarter wise to ensure this does not fall under Simpson's paradox. Hence, going a level deeper

In [10]:
# Total orders, sales, products, quantities: quarterly through 2014-2017
orders_rev_qtr = (orders_agg.groupby(["year", "quarter"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "quarter"])["products"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["quantity"].sum(), 
    how="left", 
    on=["year", "quarter"]).join(
    orders_agg.groupby(["year", "quarter"])["revenue"].sum(), 
    how="left", 
    on=["year", "quarter"])
orders_rev_qtr["%_change"] = round(orders_rev_qtr["revenue"].pct_change(),2)
orders_rev_qtr

,year,quarter,order_id,products,quantity,revenue,%_change
0,2014,1,185,385,1438,96498.7200,NaN
1,2014,2,208,405,1516,83355.5086,-0.14
2,2014,3,257,545,2097,139306.0173,0.67
3,2014,4,319,657,2528,164805.8802,0.18
4,2015,1,186,342,1301,90952.3496,-0.45
5,2015,2,248,488,1788,97852.8812,0.08
6,2015,3,275,588,2273,145554.2330,0.49
7,2015,4,329,684,2617,136173.0452,-0.06
8,2016,1,235,473,1782,136898.6390,0.01
9,2016,2,308,637,2394,149148.5428,0.09


In [11]:
print("Typical increase in revenue each quarter", round(orders_rev_qtr["%_change"].median(),2))

Typical increase in revenue each quarter 0.04


There seems to have been significant drop in revenue every third quarter as a repeating cycle through the years in comparison to the typical expected increase of ~ 4% every quarter.

Digging a level deeper - Month wise

In [12]:
# Total orders, sales, products, quantities: monthly through 2014-2017
orders_rev_month = round((orders_agg.groupby(["year", "month_num", "month"])["order_id"].count().reset_index()).join(
    orders_agg.groupby(["year", "month_num", "month"])["products"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["quantity"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]).join(
    orders_agg.groupby(["year", "month_num", "month"])["revenue"].sum(), 
    how="left", 
    on=["year", "month_num", "month"]),2).sort_values(["month_num", "year"])

orders_rev_month_pct_change = orders_rev_month.pivot(
    index=["month_num","month"],
    columns = "year",
    values = "revenue"
).reset_index()
orders_rev_month_pct_change.columns = ["month_num", "month", "2014", "2015", "2016", "2017"]
orders_rev_month_pct_change.index = orders_rev_month_pct_change["month"]
round(orders_rev_month_pct_change[[ "2014", "2015", "2016", "2017"]].pct_change(axis=1)*100,2)


,2014,2015,2016,2017
month,,,,
January,NaN,1.36,29.65,70.14
February,NaN,62.66,137.54,1.57
March,NaN,-25.41,21.37,50.72
April,NaN,55.79,18.75,-13.54
May,NaN,4.37,110.01,-37.07
June,NaN,-1.45,35.10,22.44
July,NaN,-18.71,48.88,27.14
August,NaN,32.33,-7.49,63.30
September,NaN,0.94,-37.08,76.64


In [13]:
print("Typically the order value for each year is: \n", (round(orders_agg.groupby(
    ["year"])["revenue"].median(),2)))

Typically the order value for each year is: 
 year
2014    155.37
2015    162.07
2016    145.50
2017    148.26
Name: revenue, dtype: float64


After digging a level deeper, it was observed that -on the products, quantities and orders level it seems like a steady growth.
 
And, the typical* order value for each year has seen a slight drop. Indicating the company is selling more (in reference to orders placed, quantities & products sold had been steadily increasing) but still earning less revenue per order.

*Median order value was used instead of average order value to reduce the influence of unusually large orders

#### Hypothesis 1: Median Order Value (AOV) is declining because customers buy cheaper products

In [14]:
print("The typical order value is:", round( orders_agg["revenue"].median(),2))

The typical order value is: 151.96


In [15]:
# Year-wise Median Order value
round(orders_agg.groupby("year")["revenue"].median(), 2)

year
2014    155.37
2015    162.07
2016    145.50
2017    148.26
Name: revenue, dtype: float64

In [16]:
# Monthly median order value for each year
monthly_ov = round(orders_agg.groupby(["month_num","month","year"])["revenue"].median(), 2).reset_index()
monthly_ov.columns = ["month_num", "month", "year", "order_value"]
monthly_ov.pivot(
    index= ["month_num", "month"],
    columns="year",
    values = "order_value"
).reset_index()

year,month_num,month,2014,2015,2016,2017
0,1,January,222.29,139.42,126.18,200.91
1,2,February,90.61,148.76,178.97,159.73
2,3,March,154.21,136.92,197.76,168.27
3,4,April,113.62,192.22,145.48,81.18
4,5,May,151.95,95.61,167.84,165.64
5,6,June,139.80,173.94,198.67,175.32
6,7,July,171.22,195.47,107.83,172.42
7,8,August,164.48,146.74,146.69,195.62
8,9,September,106.94,170.14,121.78,159.12
9,10,October,264.92,136.72,152.17,114.38


In [17]:
# Quarter-wise median order value
round(orders_agg.groupby(["year","quarter"])["revenue"].median(), 2).reset_index().pivot(
    index= ["year"],
    columns="quarter",
    values = "revenue"
)

quarter,1,2,3,4
year,,,,
2014,133.13,138.38,145.57,194.32
2015,138.17,146.36,167.86,182.91
2016,167.89,159.39,124.58,143.96
2017,170.92,129.83,163.76,132.52


From 2014-2015 the quarterly median order value had been steadily increasing. However since 2016, the quarterly median order value had been decreasing

The analysis, supports the hypothesis. Although the number of orders kept increasing each year, the Median Order Value consistently declined. This suggests that customers were placing lower-value orders over time, which likely contributed to the inconsistent growth in total revenue.

#### Hypothesis 2: There are specific categories that are causing decline in median order value

- revenue, orders, quantity, median order value, growth%

sub-category - furnityure, technology, office supplies

In [61]:
# Products, categories and revenue generated throughout the year
products_rev = orders[["Product ID", "Order Date","Sales"]].merge(
    dates[["date","year","month_num","month"]], 
    left_on="Order Date", right_on="date").merge(
        product[["product_id","Product ID", "Category", 
                 "Sub-Category"]], 
        on="Product ID" ).sort_values(
                            by=["year","month_num"], ascending=True
        )[["year","month_num","month",
                           "product_id","Product ID","Category",
                           "Sub-Category", "Sales"]]
#products_rev.index= products_rev["month_num"]
#products_rev
products_rev = products_rev[["year", "Product ID", "Category", "Sub-Category", "Sales" ]]
products_rev.columns = ["year", "product_id", "category", "sub_category", "revenue"]
products_rev = round(products_rev.groupby(["year","product_id"])["revenue"].median().reset_index(),2).head()
products_rev.head()

,year,product_id,revenue
0,2014,FUR-BO-10000330,411.33
1,2014,FUR-BO-10000362,341.96
2,2014,FUR-BO-10000468,155.46
3,2014,FUR-BO-10000711,425.88
4,2014,FUR-BO-10001337,335.72


In [70]:
# Revenues generated by old products (that were released until 2016) throughout the years 2014-2017

products_rev_pct_change = round(products_rev.groupby(["year","product_id"])["revenue"].sum().reset_index().pivot(
    index="product_id",
    columns = "year",
    values = "revenue"
).pct_change(axis=1),2).reset_index()
products_rev_pct_change.head()


year,product_id,2014
0,FUR-BO-10000330,NaN
1,FUR-BO-10000362,NaN
2,FUR-BO-10000468,NaN
3,FUR-BO-10000711,NaN
4,FUR-BO-10001337,NaN


In [57]:
# Products existing more than 3 years and their revenue trend

products_declined = products_rev_pct_change[~products_rev_pct_change[2017].isnull()][~products_rev_pct_change[2016].isnull()][products_rev_pct_change[2017] < 0.0].sort_values(
    by=2017, ascending=True
).reset_index(drop=True)
products_declined.head()

C:\Users\Tejali\AppData\Local\Temp\ipykernel_21288\3147428678.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  products_declined = products_rev_pct_change[~products_rev_pct_change[2017].isnull()][~products_rev_pct_change[2016].isnull()][products_rev_pct_change[2017] < 0.0].sort_values(
C:\Users\Tejali\AppData\Local\Temp\ipykernel_21288\3147428678.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  products_declined = products_rev_pct_change[~products_rev_pct_change[2017].isnull()][~products_rev_pct_change[2016].isnull()][products_rev_pct_change[2017] < 0.0].sort_values(


year,product_id,2014,2015,2016,2017
0,OFF-BI-10003727,NaN,-0.48,4.52,-0.96
1,OFF-AR-10002399,NaN,0.33,3.50,-0.94
2,OFF-BI-10000545,NaN,NaN,1.38,-0.94
3,OFF-PA-10001950,NaN,NaN,-0.03,-0.93
4,OFF-BI-10002557,NaN,12.33,2.68,-0.90


In [59]:
# Products with revenue and category details
products_rev_detail = products_rev.join(
    product[["Product ID","Category","Sub-Category"]], 
    how="left").reset_index()[["year","product_id","Category","Sub-Category","revenue"]]
products_rev_detail.head()

,year,product_id,Category,Sub-Category,revenue
0,2014,FUR-CH-10004063,Office Supplies,Art,457.568
1,2014,FUR-CH-10004063,Office Supplies,Storage,2001.860
2,2014,OFF-ST-10002276,NaN,NaN,166.720
3,2014,OFF-PA-10004082,Technology,Phones,47.880
4,2014,OFF-AP-10002945,Technology,Accessories,1503.250


In [63]:
# Category level median order value change
round(products_rev_detail.groupby(["year","Category"])["revenue"].median().reset_index().pivot(
    index="Category", columns = "year", values = "revenue"
).pct_change(axis=1),2)

year,2014,2015,2016,2017
Category,,,,
Furniture,NaN,0.29,-0.42,-0.21
Office Supplies,NaN,-0.34,0.30,-0.13
Technology,NaN,0.41,-0.24,0.10


In [ ]:
# Sub-category level revenue change
round(products_rev_detail.groupby(["year","Category","Sub-Category"])["revenue"].sum().reset_index().pivot(
    index=["Category","Sub-Category"], columns = "year", values = "revenue").pct_change(axis=1),2).reset_index().sort_values(2017, ascending=True)

year,Category,Sub-Category,2014,2015,2016,2017
14,Technology,Copiers,NaN,7.14,-0.91,-0.99
11,Office Supplies,Storage,NaN,-0.44,-0.78,-0.96
8,Office Supplies,Fasteners,NaN,-0.63,-0.66,-0.90
1,Furniture,Chairs,NaN,-0.87,-0.51,-0.88
0,Furniture,Bookcases,NaN,-0.01,-0.08,-0.85
5,Office Supplies,Art,NaN,-0.30,0.66,-0.79
4,Office Supplies,Appliances,NaN,0.29,-0.68,-0.73
16,Technology,Phones,NaN,-0.46,-0.40,-0.71
9,Office Supplies,Labels,NaN,-0.74,0.59,-0.69
2,Furniture,Furnishings,NaN,-0.51,0.00,-0.57


#### Hypothesis 3: There has been an increase in discount provided across products

#### Hypothesis 4: There has been a decline in revenue due to the underperformance of certain regions


region, state, city

##### Region

In [102]:
# Revenue change across regions through 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["Region", 
                                     "year"])["revenue"].sum().reset_index().pivot(
                                         index = "Region",
                                         columns="year",
                                         values="revenue"
                                     ).pct_change(axis = 1),4)

year,2014,2015,2016,2017
Region,,,,
Central,NaN,-0.2019,0.4269,0.0343
East,NaN,-0.0909,0.1499,0.3663
South,NaN,0.7351,-0.1415,0.3999
West,NaN,-0.0883,0.6056,-0.0036


In [100]:
# Median order value across regions through years 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["Region", 
                                     "year"])["revenue"].median().reset_index().pivot(
                                         index = "Region",
                                         columns="year",
                                         values="revenue"
                                     ), 2)

year,2014,2015,2016,2017
Region,,,,
Central,139.65,143.52,126.18,149.54
East,199.94,157.13,151.51,160.65
South,152.64,199.33,133.38,148.88
West,148.12,158.67,167.18,132.52


##### State

In [106]:
# Revenue change across states through 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["State", 
                                     "year"])["revenue"].sum().reset_index().pivot(
                                         index = "State",
                                         columns="year",
                                         values="revenue"
                                     ).pct_change(axis = 1),4).sort_values(2017, ascending=True)

year,2014,2015,2016,2017
State,,,,
New Mexico,NaN,3.3842,0.6843,-0.9240
Colorado,NaN,-0.3963,0.4137,-0.6556
Oregon,NaN,0.0842,0.5390,-0.6543
Montana,NaN,10.6817,-0.5088,-0.6412
Mississippi,NaN,0.1873,-0.0701,-0.6303
Missouri,NaN,-0.5312,18.3932,-0.6272
Minnesota,NaN,-0.4802,0.6420,-0.6237
North Dakota,NaN,NaN,NaN,-0.6190
Arkansas,NaN,2.4860,2.9562,-0.6024


In [114]:
# Median order value across states through 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["State", 
                                     "year"])["revenue"].median().reset_index().pivot(
                                         index = "State",
                                         columns="year",
                                         values="revenue"
                                     ),2)

year,2014,2015,2016,2017
State,,,,
Alabama,344.27,106.68,373.09,450.99
Arizona,152.82,258.02,342.76,136.61
Arkansas,92.52,90.17,327.42,258.03
California,137.57,154.81,133.54,148.26
Colorado,83.73,225.93,270.73,109.90
Connecticut,182.89,74.52,104.05,369.54
Delaware,129.03,467.65,71.09,547.30
District of Columbia,9.96,NaN,1376.44,NaN
Florida,132.26,167.00,142.18,151.89


##### City

In [118]:
# Revenue change across cities through 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["City", 
                                     "year"])["revenue"].sum().reset_index().pivot(
                                         index = "City",
                                         columns="year",
                                         values="revenue"
                                     ).pct_change(axis=1),2)

year,2014,2015,2016,2017
City,,,,
Aberdeen,NaN,NaN,NaN,NaN
Abilene,NaN,NaN,NaN,NaN
Akron,NaN,8.91,-0.88,4.42
Albuquerque,NaN,NaN,0.47,-0.61
Alexandria,NaN,2.10,-0.94,NaN
...,...,...,...,...
Woonsocket,NaN,NaN,-0.57,NaN
Yonkers,NaN,6.63,3.12,-0.98
York,NaN,NaN,NaN,NaN


In [120]:
# Median Order Value across cities through 2014-2017
round(orders_agg.join(
    customer[["Customer ID",
               "Region", "Country", 
                "State", "City"]], 
                how="left").groupby(["City", 
                                     "year"])["revenue"].median().reset_index().pivot(
                                         index = "City",
                                         columns="year",
                                         values="revenue"
                                     ),2)

year,2014,2015,2016,2017
City,,,,
Aberdeen,NaN,NaN,NaN,437.65
Abilene,NaN,NaN,NaN,604.77
Akron,67.63,669.95,34.38,387.99
Albuquerque,NaN,214.78,316.30,68.42
Alexandria,602.43,288.00,53.38,NaN
...,...,...,...,...
Woonsocket,NaN,1349.63,290.77,NaN
Yonkers,25.95,98.97,407.88,13.92
York,NaN,235.97,NaN,17.48


#### Hypothesis 5: The reasoning for revenue decline is due to certain customer segments generating lower order values

consumer, coporate, home office

#### Hypothesis 5: The reasoning for revenue decline is due to certain products lower order values

top products, worst products, products losing revenue, products declining MOV

#### Products driving the most revenue - Pareto analysis

In [ ]:
pareto_product_analysis = pd.DataFrame(round(orders.groupby("Product ID")["Sales"].sum(),2).sort_values(ascending = False).reset_index())
pareto_product_analysis["cumulative_revenue"] = round(pareto_product_analysis["Sales"].cumsum(), 2)
pareto_product_analysis["cumulative_%"] = round((pareto_product_analysis["cumulative_revenue"]/
                                           pareto_product_analysis["Sales"].sum())*100,2)
pareto_product_analysis[pareto_product_analysis["cumulative_%"]<=80]


It can be observed that around 22% of top products (based on revenue) contribute to 80% of revenue

How are different regions performing in terms of revenue

In [ ]:
region_sales = orders[["Customer ID", 
        "Order ID", "Order Date",
        "Sales"]].merge(customer, 
                        on="Customer ID").merge(dates, left_on="Order Date",right_on="date").groupby(
                            ["year","Region"]
                        )["Sales"].sum().reset_index().pivot(
    index = "year",
    columns=["Region"], 
    values = "Sales").reset_index()
region_sales["%_change_central"] = round(region_sales["Central"].pct_change()*100, 2)
region_sales["%_change_east"] = round(region_sales["East"].pct_change()*100, 2)
region_sales["%_change_south"] = round(region_sales["South"].pct_change()*100, 2)
region_sales["%_change_west"] = round(region_sales["West"].pct_change()*100, 2)
region_sales[["year","Central","%_change_central","South","%_change_south","West","%_change_west","East","%_change_east"]]

It can be observed that there had been a major drop in revenue in Central region from 37.56% to 6.49% in year 2017

Identifying which customers buy the most

In [ ]:
customer_sales = round(orders.groupby("Customer ID")["Sales"].sum(),2).sort_values(ascending = False).reset_index()
customer_sales["pct_contribution"] = round(customer_sales['Sales'] *100/ customer_sales['Sales'].sum(),2)
customer_sales

Monthly sales performance

In [ ]:
monthly_perf_2017 = orders_agg.groupby(["year","month_num","month"])["revenue"].sum().reset_index()
monthly_perf_2017["%_change_rev"] = round(monthly_perf_2017["revenue"].pct_change(),2)
monthly_perf_2017